In [11]:
# =========================================
# Breast Cancer Dataset (ML)
# =========================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score


# =========================================
# Load dataset WITHOUT header
# =========================================

df = pd.read_csv('BreastCancerWc.csv', header=None)

# Assign simple column names
df.columns = ['id', 'clump', 'size', 'shape', 'adhesion',
              'epithelial', 'nuclei', 'chromatin',
              'nucleoli', 'mitoses', 'class']

print("Original Data:")
print(df.head())


# =========================================
# i. Data Cleaning
# =========================================

# Drop ID column (not useful)
df.drop('id', axis=1, inplace=True)

# Replace '?' with NaN
df.replace('?', np.nan, inplace=True)

# Remove missing values
df.dropna(inplace=True)

# Convert to numeric
df = df.apply(pd.to_numeric)

print("\nAfter Cleaning:")
print(df.head())


# =========================================
# j. Outlier Removal
# =========================================

Q1 = df.quantile(0.25)
Q3 = df.quantile(0.75)
IQR = Q3 - Q1

df = df[~((df < (Q1 - 1.5 * IQR)) | 
          (df > (Q3 + 1.5 * IQR))).any(axis=1)]

print("\nAfter Outlier Removal:")
print(df.head())


# =========================================
# k. Data Transformation
# =========================================

# Convert class values:
# 2 → benign (0)
# 4 → malignant (1)
df['class'] = df['class'].map({2: 0, 4: 1})

print("\nAfter Transformation:")
print(df['class'].value_counts())


# =========================================
# l. Model Building
# =========================================

X = df.drop('class', axis=1)
y = df['class']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)


# Logistic Regression
lr = LogisticRegression(max_iter=200)
lr.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test)
acc_lr = accuracy_score(y_test, y_pred_lr)

print("\nLogistic Regression Accuracy:", acc_lr)


# Naive Bayes
nb = GaussianNB()
nb.fit(X_train, y_train)

y_pred_nb = nb.predict(X_test)
acc_nb = accuracy_score(y_test, y_pred_nb)

print("Naive Bayes Accuracy:", acc_nb)


# Compare
if acc_lr > acc_nb:
    print("\nLogistic Regression is better")
else:
    print("\nNaive Bayes is better")

Original Data:
        id  clump  size  shape  adhesion  epithelial nuclei  chromatin  \
0  1000025      5     1      1         1           2      1          3   
1  1002945      5     4      4         5           7     10          3   
2  1015425      3     1      1         1           2      2          3   
3  1016277      6     8      8         1           3      4          3   
4  1017023      4     1      1         3           2      1          3   

   nucleoli  mitoses  class  
0         1        1      2  
1         2        1      2  
2         1        1      2  
3         7        1      2  
4         1        1      2  

After Cleaning:
   clump  size  shape  adhesion  epithelial  nuclei  chromatin  nucleoli  \
0      5     1      1         1           2       1          3         1   
1      5     4      4         5           7      10          3         2   
2      3     1      1         1           2       2          3         1   
3      6     8      8         1        